In [2]:
"""
BBL 2026 Final — Data Exploration Script
==========================================
Run this script to explore the coded match data.
Each section presents data in a readable format.
Read the output and draw your own insights.
 
USAGE
-----
python notebooks/explore.py
"""
 
import pandas as pd
import numpy as np
 
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.1f}".format)

In [3]:
# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv("/Users/kushgirap/Desktop/bbl-2026-analysis/data/bbl_2026_final_coded - Sheet3.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.rename(columns={"ball_speed_(kmph)": "ball_speed"})
df["extras"]    = df["extras"].fillna(0)
df["ball_speed"] = pd.to_numeric(df["ball_speed"], errors="coerce")
 
# Boolean helpers
df["is_wicket"]   = df["wicket"]  == "Yes"
df["is_wide"]     = df["wide"]    == "Yes"
df["is_dot"]      = (df["runs_batter"] == 0) & ~df["is_wide"]
df["is_boundary"] = df["runs_batter"].isin([4, 6])
df["is_four"]     = df["runs_batter"] == 4
df["is_six"]      = df["runs_batter"] == 6
df["is_legal"]    = (df["is_wide"]==False) & (df["no_ball"] == "No")
df['runs_total'] = df['runs_batter'] + df['extras']
 
# Phase
def assign_phase(over):
    if over <= 5:    return "Powerplay (1-6)"
    elif over <= 14: return "Middle (7-15)"
    else:            return "Death (16-20)"
df["phase"] = df["over"].apply(assign_phase)
 
# Direction zones
direction_map = {
    "Fine Leg": "Fine Leg", "Deep Fine Leg": "Fine Leg",
    "Square Leg": "Square Leg", "Deep Square Leg": "Square Leg",
    "Deep Backward Square Leg": "Fine Leg", "Forward Square Leg": "Mid Wicket",
    "Deep Forward Square Leg": "Mid Wicket", "Mid Wicket": "Mid Wicket",
    "Deep Mid Wicket": "Mid Wicket", "Mid On": "Mid On",
    "Long On": "Mid On", "Straight Hit": "Straight",
    "Long Off": "Mid Off", "Mid Off": "Mid Off",
    "Extra Cover": "Cover", "Deep Cover": "Cover",
    "Cover": "Cover", "Sweeper Cover": "Cover",
    "Point": "Point", "Deep Forward Point": "Point",
    "Backward Point": "Point", "Deep Backward Point": "Point",
    "Gully": "Point", "Third Man": "Third Man",
    "Fine Third Man": "Third Man", "Deep Third Man": "Third Man",
}
df["direction_zone"] = df["actual_direction"].map(direction_map)

In [4]:
df.columns

Index(['innings', 'over', 'ball', 'striker', 'non_striker', 'bowler', 'type_of_bowler', 'bowling_end', 'runs_batter',
       'extras', 'wide', 'no_ball', 'leg_bye', 'bye', 'wicket', 'dismissal_type', 'caught_at', 'fielder', 'line',
       'length', 'shot_played', 'intended_direction', 'actual_direction', 'reason_for_mismatch', 'middled', 'edged',
       'beaten', 'ball_speed', 'batter_type', 'is_wicket', 'is_wide', 'is_dot', 'is_boundary', 'is_four', 'is_six',
       'is_legal', 'runs_total', 'phase', 'direction_zone'],
      dtype='object')

In [5]:
df["striker"].unique()

array(['Matthew Gilkes', 'David Warner', 'Sam Konstas', 'Sam Billings',
       'Nic Maddinson', 'Chris Green ', 'Daniel Sams', 'Babar Azam',
       'Steve Smith', 'Josh Phillipe', 'Moises Henriques', 'Sam Curran',
       'Lachlan Shaw', 'Jack Edwards'], dtype=object)

In [6]:
df["bowler"].unique()

array(['Sean Abbott', 'Mitchell Starc', 'Jack Edwards', 'Ben Manenti',
       'Joel Davis', 'Sam Curran', 'Chris Green ', 'Ryan Hadley',
       'Wes Agar', 'Nathan McAndrew', "Aidan O'Connor", 'Tanveer Sangha',
       'Nic Maddinson'], dtype=object)

In [7]:
# Team mapping — update with actual squads
THUNDER  = ['Matthew Gilkes', 'David Warner', 'Sam Konstas', 'Sam Billings',
       'Nic Maddinson', 'Chris Green ', 'Daniel Sams', "Aidan O'Connor", 'Tanveer Sangha','Ryan Hadley',
       'Wes Agar']
SIXERS   = ['Babar Azam',
       'Steve Smith', 'Josh Phillipe', 'Moises Henriques', 'Sam Curran',
       'Lachlan Shaw', 'Jack Edwards', "Sean Abbott", "Mitchell Starc", "Ben Manenti",
            "Joel Davis"]
 
df["batting_team"] = df["striker"].apply(
    lambda x: "Thunder" if x in THUNDER else "Sixers" if x in SIXERS else "Unknown"
)
df["bowling_team"] = df["bowler"].apply(
    lambda x: "Thunder" if x in THUNDER else "Sixers" if x in SIXERS else "Unknown"
)

In [8]:
# ── Helper functions ──────────────────────────────────────────────────────────
def divider(title):
    print("\n" + "═" * 70)
    print(f"  {title}")
    print("═" * 70)
 
def sub(title):
    print(f"\n── {title} " + "─" * (60 - len(title)))
 
def show(df, title=""):
    if title:
        print(f"\n{title}")
    print(df.to_string())
    print()

In [9]:
# SECTION 1 — MATCH OVERVIEW
# ══════════════════════════════════════════════════════════════════════════════
divider("SECTION 1 — MATCH OVERVIEW")
 
sub("Innings summary")

# Runs — include all deliveries including extras
runs_summary = (
    df.groupby(["innings", "batting_team"])
    .agg(
        runs       = ("runs_batter", "sum"),
        extras     = ("extras", "sum"),
        boundaries = ("is_boundary", "sum"),
        fours      = ("is_four", "sum"),
        sixes      = ("is_six", "sum"),
        wickets    = ("is_wicket", "sum"),
    )
)
runs_summary["total_runs"] = runs_summary["runs"] + runs_summary["extras"]

# Balls — only legal deliveries
balls_summary = (
    df[df["is_legal"]].groupby(["innings", "batting_team"])
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)

# Combine
innings_summary = runs_summary.join(balls_summary)
innings_summary["run_rate"] = (innings_summary["total_runs"] / (innings_summary["balls"] / 6)).round(2)
innings_summary["dot_pct"]  = (innings_summary["dots"] / innings_summary["balls"] * 100).round(1)
innings_summary["bdry_pct"] = (innings_summary["boundaries"] / innings_summary["balls"] * 100).round(1)

show(innings_summary)
 
sub("Phase breakdown — both innings")
phase_summary = (
    df[df["is_legal"]].groupby(["innings", "batting_team", "phase"])
    .agg(
        balls    = ("ball", "count"),
        runs     = ("runs_batter", "sum"),
        wickets  = ("is_wicket", "sum"),
        dots     = ("is_dot", "sum"),
        boundaries = ("is_boundary", "sum"),
    )
    .assign(
        run_rate = lambda x: (x["runs"] / (x["balls"] / 6)).round(2),
        dot_pct  = lambda x: (x["dots"] / x["balls"] * 100).round(1),
    )
)
show(phase_summary)
 


══════════════════════════════════════════════════════════════════════
  SECTION 1 — MATCH OVERVIEW
══════════════════════════════════════════════════════════════════════

── Innings summary ─────────────────────────────────────────────
                      runs  extras  boundaries  fours  sixes  wickets  total_runs  balls  dots  run_rate  dot_pct  bdry_pct
innings batting_team                                                                                                       
1       Thunder        178    11.0          24     17      7        6       189.0    120    43       9.4     35.8      20.0
2       Sixers         185     6.0          26     14     12        5       191.0    104    32      11.0     30.8      25.0


── Phase breakdown — both innings ──────────────────────────────
                                      balls  runs  wickets  dots  boundaries  run_rate  dot_pct
innings batting_team phase                                                                     
1      

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — BATTING
# ══════════════════════════════════════════════════════════════════════════════
divider("SECTION 2 — BATTING")

sub("Individual batter scorecard")
# Runs from all deliveries
batter_runs = (
    df.groupby(["innings", "batting_team", "striker"])
    .agg(
        runs       = ("runs_batter", "sum"),
        boundaries = ("is_boundary", "sum"),
        fours      = ("is_four", "sum"),
        sixes      = ("is_six", "sum"),
        wickets    = ("is_wicket", "sum"),
    )
)
# Balls and dots from legal deliveries only
batter_balls = (
    df[df["is_legal"]].groupby(["innings", "batting_team", "striker"])
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)
batter_card = batter_runs.join(batter_balls).reset_index()
batter_card["strike_rate"] = (batter_card["runs"] / batter_card["balls"] * 100).round(1)
batter_card["dot_pct"]     = (batter_card["dots"] / batter_card["balls"] * 100).round(1)
batter_card = batter_card.sort_values(["innings", "runs"], ascending=[True, False])
show(batter_card)

sub("Batter vs bowler type — SR and dot% (min 6 balls)")
# Runs from all deliveries
bvt_runs = (
    df.groupby(["striker", "type_of_bowler", "bowler"])
    .agg(
        runs       = ("runs_batter", "sum"),
        wickets    = ("is_wicket", "sum"),
        boundaries = ("is_boundary", "sum"),
    )
)
# Balls and dots from legal deliveries only
bvt_balls = (
    df[df["is_legal"]].groupby(["striker", "type_of_bowler", "bowler"])
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)
bvt = bvt_runs.join(bvt_balls).reset_index()
bvt["strike_rate"] = (bvt["runs"] / bvt["balls"] * 100).round(1)
bvt["dot_pct"]     = (bvt["dots"] / bvt["balls"] * 100).round(1)
show(bvt[bvt["balls"] >= 6].sort_values(["striker", "dot_pct"], ascending=[True, False]))

sub("Boundary zones — where did scoring happen?")
zone = (
    df[df["is_boundary"]].groupby("direction_zone")
    .agg(
        boundaries = ("is_boundary", "sum"),
        fours      = ("is_four", "sum"),
        sixes      = ("is_six", "sum"),
    )
    .assign(pct=lambda x: (x["boundaries"] / x["boundaries"].sum() * 100).round(1))
    .sort_values("boundaries", ascending=False)
)
show(zone)

sub("Boundary zones by innings")
zone_inn = (
    df[df["is_boundary"]].groupby(["innings", "batting_team", "direction_zone"])
    .agg(boundaries=("is_boundary", "sum"))
    .assign(pct=lambda x: (
        x["boundaries"] / x.groupby(level=[0,1])["boundaries"].transform("sum") * 100
    ).round(1))
    .sort_values(["innings", "boundaries"], ascending=[True, False])
)
show(zone_inn)

sub("Shot success — runs, dots, wickets and risk per shot type (min 5 balls)")
# Runs from all deliveries
shot_runs = (
    df.groupby("shot_played")
    .agg(
        runs       = ("runs_batter", "sum"),
        wickets    = ("is_wicket", "sum"),
        boundaries = ("is_boundary", "sum"),
        beaten     = ("beaten", lambda x: (x == "Yes").sum()),
        edged      = ("edged", lambda x: (x == "Yes").sum()),
    )
)
# Balls and dots from legal deliveries only
shot_balls = (
    df[df["is_legal"]].groupby("shot_played")
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)
shot_analysis = shot_runs.join(shot_balls).reset_index()
shot_analysis["strike_rate"] = (shot_analysis["runs"] / shot_analysis["balls"] * 100).round(1)
shot_analysis["dot_pct"]     = (shot_analysis["dots"] / shot_analysis["balls"] * 100).round(1)
shot_analysis["risk_factor"] = ((shot_analysis["wickets"] + shot_analysis["beaten"]) / shot_analysis["balls"] * 100).round(1)
shot_analysis = shot_analysis.sort_values("balls", ascending=False)
show(shot_analysis[shot_analysis["balls"] >= 5])
a



══════════════════════════════════════════════════════════════════════
  SECTION 2 — BATTING
══════════════════════════════════════════════════════════════════════

── Individual batter scorecard ─────────────────────────────────
    innings batting_team           striker  runs  boundaries  fours  sixes  wickets  balls  dots  strike_rate  dot_pct
2         1      Thunder      David Warner   110          15     11      4        0     63    18        174.6     28.6
4         1      Thunder     Nic Maddinson    26           4      2      2        1     16     7        162.5     43.8
5         1      Thunder      Sam Billings    14           1      1      0        1     13     4        107.7     30.8
3         1      Thunder    Matthew Gilkes    12           2      2      0        1     12     6        100.0     50.0
1         1      Thunder       Daniel Sams    10           2      1      1        1      4     2        250.0     50.0
6         1      Thunder       Sam Konstas     6       

NameError: name 'a' is not defined

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — BOWLING
# ══════════════════════════════════════════════════════════════════════════════
divider("SECTION 3 — BOWLING")

sub("Bowler summary — economy, wickets, beat bat, pressure")
# Runs from all deliveries
bowler_runs = (
    df.groupby(["innings", "bowling_team", "bowler"])
    .agg(
        runs       = ("runs_batter", "sum"),
        extras     = ("extras", "sum"),
        wickets    = ("is_wicket", "sum"),
        boundaries = ("is_boundary", "sum"),
        beat_bat   = ("beaten", lambda x: (x == "Yes").sum()),
        edges      = ("edged", lambda x: (x == "Yes").sum()),
        wides      = ("is_wide", "sum"),
    )
)
bowler_runs["total_runs"] = bowler_runs["runs"] + bowler_runs["extras"]

# Balls and dots from legal deliveries only
bowler_balls = (
    df[df["is_legal"]].groupby(["innings", "bowling_team", "bowler"])
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)
bowler_summary = bowler_runs.join(bowler_balls).reset_index()
bowler_summary["overs"]        = (bowler_summary["balls"] / 6).round(1)
bowler_summary["economy"]      = (bowler_summary["total_runs"] / (bowler_summary["balls"] / 6)).round(2)
bowler_summary["dot_pct"]      = (bowler_summary["dots"] / bowler_summary["balls"] * 100).round(1)
bowler_summary["beat_bat_pct"] = (bowler_summary["beat_bat"] / bowler_summary["balls"] * 100).round(1)
bowler_summary["wkt_to_bdry"]  = (bowler_summary["wickets"] / (bowler_summary["boundaries"] + 1)).round(2)
bowler_summary = bowler_summary.sort_values(["innings", "economy"])
show(bowler_summary)

sub("Line and length effectiveness — dot%, economy, wickets (min 5 balls)")
# Runs from all deliveries
ll_runs = (
    df.groupby(["line", "length"])
    .agg(
        runs       = ("runs_batter", "sum"),
        extras     = ("extras", "sum"),
        wickets    = ("is_wicket", "sum"),
        boundaries = ("is_boundary", "sum"),
        beat_bat   = ("beaten", lambda x: (x == "Yes").sum()),
    )
)
ll_runs["total_runs"] = ll_runs["runs"] + ll_runs["extras"]

# Balls and dots from legal only
ll_balls = (
    df[df["is_legal"]].groupby(["line", "length"])
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)
ll = ll_runs.join(ll_balls).reset_index()
ll["economy"] = (ll["total_runs"] / (ll["balls"] / 6)).round(2)
ll["dot_pct"] = (ll["dots"] / ll["balls"] * 100).round(1)
ll = ll.sort_values("dot_pct", ascending=False)
show(ll[ll["balls"] >= 5])

sub("Line and length by innings — where did each team bowl?")
ll_inn_runs = (
    df.groupby(["innings", "bowling_team", "line", "length"])
    .agg(
        runs    = ("runs_batter", "sum"),
        extras  = ("extras", "sum"),
        wickets = ("is_wicket", "sum"),
    )
)
ll_inn_runs["total_runs"] = ll_inn_runs["runs"] + ll_inn_runs["extras"]

ll_inn_balls = (
    df[df["is_legal"]].groupby(["innings", "bowling_team", "line", "length"])
    .agg(
        balls = ("ball", "count"),
        dots  = ("is_dot", "sum"),
    )
)
ll_inn = ll_inn_runs.join(ll_inn_balls).reset_index()
ll_inn["economy"] = (ll_inn["total_runs"] / (ll_inn["balls"] / 6)).round(2)
ll_inn["dot_pct"] = (ll_inn["dots"] / ll_inn["balls"] * 100).round(1)
ll_inn = ll_inn.sort_values(["innings", "dot_pct"], ascending=[True, False])
show(ll_inn[ll_inn["balls"] >= 5])

sub("Wicket-taking deliveries — line, length, dismissal type")
wickets = (
    df[df["is_wicket"]].groupby(["line", "length", "dismissal_type"])
    .size()
    .reset_index(name="wickets")
    .sort_values("wickets", ascending=False)
)
show(wickets)

sub("Pressure sequences — 3+ consecutive dot balls by bowler")
df_sorted = df.sort_values(["innings", "over", "ball"]).copy()
df_sorted["prev_dot1"] = df_sorted.groupby(["innings", "bowler"])["is_dot"].shift(1)
df_sorted["prev_dot2"] = df_sorted.groupby(["innings", "bowler"])["is_dot"].shift(2)
df_sorted["pressure"]  = (
    (df_sorted["is_dot"] == True) &
    (df_sorted["prev_dot1"] == True) &
    (df_sorted["prev_dot2"] == True)
)
pressure = (
    df_sorted.groupby(["bowling_team", "bowler"])["pressure"]
    .sum()
    .reset_index(name="pressure_sequences")
    .sort_values("pressure_sequences", ascending=False)
)
show(pressure)


══════════════════════════════════════════════════════════════════════
  SECTION 3 — BOWLING
══════════════════════════════════════════════════════════════════════

── Bowler summary — economy, wickets, beat bat, pressure ───────
    innings bowling_team           bowler  runs  extras  wickets  boundaries  beat_bat  edges  wides  total_runs  balls  dots  overs  economy  dot_pct  beat_bat_pct  wkt_to_bdry
0         1       Sixers      Ben Manenti    17     0.0        1           1         4      1      0        17.0     18     6    3.0      5.7     33.3          22.2          0.5
4         1       Sixers       Sam Curran    25     3.0        3           3         2      1      3        28.0     24    10    4.0      7.0     41.7           8.3          0.8
3         1       Sixers   Mitchell Starc    29     2.0        1           3         3      3      2        31.0     24     9    4.0      7.8     37.5          12.5          0.2
2         1       Sixers       Joel Davis     9     0.0  

In [12]:
# SECTION 4 — MISMATCHES & VULNERABILITIES
# ══════════════════════════════════════════════════════════════════════════════
divider("SECTION 4 — MISMATCHES & VULNERABILITIES")
 
sub("Shots beaten most — by line, length and bowler")
beaten = (
    df[df["beaten"] == "Yes"]
    .groupby(["shot_played", "line", "length", "bowler"])
    .size()
    .reset_index(name="times_beaten")
    .sort_values("times_beaten", ascending=False)
)
show(beaten.head(15))
 
sub("Edges — where are false edges coming from?")
edges = (
    df[df["edged"] == "Yes"]
    .groupby(["line", "length", "shot_played", "bowler"])
    .size()
    .reset_index(name="edges")
    .sort_values("edges", ascending=False)
)
show(edges.head(15))
 
sub("Mismatch patterns — intended vs actual direction")
mismatch = (
    df[df["reason_for_mismatch"].notna() & (df["reason_for_mismatch"] != "")]
    .groupby(["line", "length", "shot_played", "reason_for_mismatch"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
show(mismatch.head(15))
 
sub("Beat bat + edges combined — most dangerous bowler-line-length combination")
danger = (
    df[(df["beaten"] == "Yes") | (df["edged"] == "Yes")]
    .groupby(["bowler", "line", "length"])
    .size()
    .reset_index(name="danger_deliveries")
    .sort_values("danger_deliveries", ascending=False)
)
show(danger.head(15))


══════════════════════════════════════════════════════════════════════
  SECTION 4 — MISMATCHES & VULNERABILITIES
══════════════════════════════════════════════════════════════════════

── Shots beaten most — by line, length and bowler ──────────────
          shot_played         line           length          bowler  times_beaten
22       Square drive  Outside Off      Good Length    Jack Edwards             2
0   Back Foot Defence          Off      Good Length     Sean Abbott             1
1   Back Foot Defence  Outside Off  Short of length        Wes Agar             1
26          Step out   Outside Off             Full     Ben Manenti             1
25          Step out   Outside Leg      Good Length      Sam Curran             1
24          Step out           Off          Bouncer        Wes Agar             1
23       Square drive  Outside Off      Good Length     Sean Abbott             1
21         Square Cut  Outside Off           Yorker     Sean Abbott             1
20        

In [13]:
# SECTION 5 — BALL SPEED
# ══════════════════════════════════════════════════════════════════════════════
divider("SECTION 5 — BALL SPEED")
 
sub("Average ball speed by bowler")
speed = (
    df[df["ball_speed"].notna()].groupby("bowler")
    .agg(
        balls     = ("ball", "count"),
        avg_speed = ("ball_speed", "mean"),
        max_speed = ("ball_speed", "max"),
        min_speed = ("ball_speed", "min"),
    )
    .assign(avg_speed=lambda x: x["avg_speed"].round(1))
    .sort_values("avg_speed", ascending=False)
)
show(speed[speed["balls"] >= 6])
 
sub("Ball speed vs outcome — does pace affect dot ball rate?")
df["speed_band"] = pd.cut(
    df["ball_speed"],
    bins=[0, 120, 130, 140, 150, 200],
    labels=["<120", "120-130", "130-140", "140-150", "150+"]
)
speed_outcome = (
    df[df["ball_speed"].notna() & df["is_legal"]].groupby("speed_band")
    .agg(
        balls      = ("ball", "count"),
        runs       = ("runs_batter", "sum"),
        dots       = ("is_dot", "sum"),
        wickets    = ("is_wicket", "sum"),
        boundaries = ("is_boundary", "sum"),
    )
    .assign(
        dot_pct  = lambda x: (x["dots"] / x["balls"] * 100).round(1),
        economy  = lambda x: (x["runs"] / (x["balls"] / 6)).round(2),
    )
)
show(speed_outcome)
 
print("\n" + "═" * 70)
print("  END OF EXPLORATION — Read the outputs and draw your insights")
print("═" * 70 + "\n")


══════════════════════════════════════════════════════════════════════
  SECTION 5 — BALL SPEED
══════════════════════════════════════════════════════════════════════

── Average ball speed by bowler ────────────────────────────────
                 balls  avg_speed  max_speed  min_speed
bowler                                                 
Mitchell Starc      22      138.1      145.3      122.6
Sean Abbott         20      131.1      139.9      111.0
Nathan McAndrew      9      130.9      142.7      125.6
Wes Agar            10      128.8      138.4      107.8
Ryan Hadley         14      128.6      142.3      109.6
Aidan O'Connor      12      128.5      135.3      120.1
Jack Edwards        16      123.2      134.8      101.1
Sam Curran          21      122.5      133.0       77.1
Chris Green         20       97.2      105.5       84.4
Ben Manenti          8       95.9      104.2       91.4
Tanveer Sangha      18       89.9       98.3       84.6
Nic Maddinson        6       89.8     

/var/folders/vx/0hn39_rd5_g696zcwndcqdx80000gn/T/ipykernel_1906/4072148660.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[df["ball_speed"].notna() & df["is_legal"]].groupby("speed_band")


In [14]:
df.column

AttributeError: 'DataFrame' object has no attribute 'column'

In [15]:
smith = df[df["striker"] == "Steve Smith"]

print(smith.groupby(["bowler", "type_of_bowler"]).agg(
    balls      = ("ball", "count"),
    runs       = ("runs_batter", "sum"),
    dots       = ("is_dot", "sum"),
    boundaries = ("is_boundary", "sum"),
    beat_bat   = ("beaten", lambda x: (x=="Yes").sum()),
).assign(sr = lambda x: (x["runs"]/x["balls"]*100).round(1)))

# Line and length breakdown against Sangha vs pace
print("\nSmith vs Sangha — line/length:")
print(smith[smith["bowler"]=="Tanveer Sangha"].groupby(["line","length"]).size())

print("\nSmith vs pace — line/length:")
print(smith[smith["type_of_bowler"].isin(["Right Medium Fast","Right Fast"])].groupby(["line","length","shot_played"]).agg(
    balls=("ball","count"),
    runs=("runs_batter","sum"),
    boundaries=("is_boundary","sum")
))

                                   balls  runs  dots  boundaries  beat_bat    sr
bowler          type_of_bowler                                                  
Aidan O'Connor  Right Medium Fast      3     3     0           0         0 100.0
Chris Green     Right Off              3     3     0           0         0 100.0
Nathan McAndrew Right Medium Fast      9    28     1           5         0 311.1
Ryan Hadley     Right Fast            10    32     1           5         0 320.0
Tanveer Sangha  Right Leg             12    12     5           1         2 100.0
Wes Agar        Right Medium Fast      9    22     2           3         1 244.4

Smith vs Sangha — line/length:
line         length         
Middle       Full               1
             Good Length        4
             Short of length    1
Off          Full               1
             Good Length        2
Outside Off  Good Length        2
             Short of length    1
dtype: int64

Smith vs pace — line/length:
          

In [16]:
warner = df[df["striker"] == "David Warner"]

print(warner.groupby(["bowler", "type_of_bowler"]).agg(
    balls      = ("ball", "count"),
    runs       = ("runs_batter", "sum"),
    dots       = ("is_dot", "sum"),
    boundaries = ("is_boundary", "sum"),
    beat_bat   = ("beaten", lambda x: (x=="Yes").sum()),
).assign(sr = lambda x: (x["runs"]/x["balls"]*100).round(1)))

# Line comparison — left arm vs right arm at Warner
print("\nWarner vs RIGHT arm — line breakdown:")
print(warner[warner["type_of_bowler"].isin(["Right Medium Fast","Right Fast"])].groupby("line").agg(
    balls=("ball","count"), runs=("runs_batter","sum"), boundaries=("is_boundary","sum")
))

print("\nWarner vs LEFT arm — line breakdown:")
print(warner[warner["type_of_bowler"].isin(["Left Fast","Left Medium Fast"])].groupby("line").agg(
    balls=("ball","count"), runs=("runs_batter","sum"), boundaries=("is_boundary","sum")
))

                                  balls  runs  dots  boundaries  beat_bat    sr
bowler         type_of_bowler                                                  
Ben Manenti    Right Off              6     6     1           0         0 100.0
Jack Edwards   Right Medium Fast     14    33     2           5         1 235.7
Joel Davis     Left Off               3     7     0           1         0 233.3
Mitchell Starc Left Fast             15    15     6           1         2 100.0
Sam Curran     Left Medium Fast      11     5     3           0         1  45.5
Sean Abbott    Right Medium Fast     22    44     7           8         5 200.0

Warner vs RIGHT arm — line breakdown:
             balls  runs  boundaries
line                                
Leg              2     2           0
Middle           8    20           3
Off              1     4           1
Outside Leg     15    32           6
Outside Off     10    19           3

Warner vs LEFT arm — line breakdown:
             balls  runs

In [17]:
# ── CURRAN DEEP DIVE ──────────────────────────────────────────────────────

curran = df[df['bowler'] == 'Sam Curran']

print("Curran — who did he bowl to:")
print(curran.groupby('striker').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
).sort_values('runs', ascending=False))

print("\nCurran — line/length breakdown:")
print(curran.groupby(['line', 'length']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
))

print("\nCurran — wicket deliveries (full detail):")
print(curran[curran['is_wicket'] == 1][
    ['striker', 'batter_type', 'line', 'length', 'shot_played',
     'dismissal_type', 'edged', 'beaten', 'ball_speed']
])

Curran — who did he bowl to:
               balls  runs  boundaries  wickets  dots
striker                                              
Daniel Sams        4    10           2        1     2
Sam Billings       4     7           1        0     0
David Warner       8     5           0        0     3
Sam Konstas        4     2           0        0     2
Nic Maddinson      3     1           0        1     2
Chris Green        1     0           0        1     1

Curran — line/length breakdown:
                             balls  runs  dots  boundaries  wickets
line        length                                                 
Leg         Bouncer              1     0     1           0        1
            Full                 1     1     0           0        0
            Good Length          1     0     1           0        1
            Short of length      2     0     2           0        0
Middle      Full                 2     2     0           0        0
            Good Length       

In [18]:
# ── HADLEY DEEP DIVE ──────────────────────────────────────────────────────

hadley = df[df['bowler'] == 'Ryan Hadley']

print("Hadley — who did he bowl to:")
print(hadley.groupby('striker').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
).sort_values('runs', ascending=False))

print("\nHadley — line/length breakdown:")
print(hadley.groupby(['line', 'length']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
))

print("\nHadley — shot by shot against him:")
print(hadley.groupby(['line', 'length', 'shot_played']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum')
).sort_values('runs', ascending=False).head(20))

print("\nHadley — by phase:")
print(hadley.groupby('phase').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    economy=('runs_batter', lambda x: round(x.sum() / (len(x)/6), 2))
))

Hadley — who did he bowl to:
              balls  runs  boundaries  wickets  dots
striker                                             
Steve Smith       8    32           5        0     1
Babar Azam        4     8           1        0     0
Jack Edwards      2     6           1        0     1

Hadley — line/length breakdown:
                             balls  runs  dots  boundaries  wickets
line        length                                                 
Leg         FT                   1     6     0           1        0
Middle      Good Length          1     6     0           1        0
            Short of length      1     1     0           0        0
Off         Good Length          1     1     0           0        0
Outside Leg Yorker               0     0     0           0        0
Outside Off Bouncer              1     0     1           0        0
            Good Length          8    26     1           4        0
            Short of length      1     6     0           1   

In [19]:
konstas = df[df['striker'] == 'Sam Konstas']

print("Konstas — overall:")
print(konstas.agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
))

print("\nKonstas — who bowled to him:")
print(konstas.groupby(['bowler', 'type_of_bowler']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
).sort_values('runs', ascending=False))

print("\nKonstas — line/length breakdown:")
print(konstas.groupby(['line', 'length']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
))

print("\nKonstas — shot selection:")
print(konstas.groupby(['line', 'length', 'shot_played']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum')
).sort_values('runs', ascending=False).head(15))

print("\nKonstas — dismissal detail:")
print(konstas[konstas['is_wicket'] == 1][
    ['bowler', 'type_of_bowler', 'line', 'length', 
     'shot_played', 'dismissal_type', 'edged', 'beaten', 'ball_speed']
])

print("\nKonstas — by phase:")
print(konstas.groupby('phase').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    dots=('is_dot', 'sum')
))

Konstas — overall:
            is_legal  runs_batter  is_boundary  is_wicket  is_dot
balls           11.0          NaN          NaN        NaN     NaN
runs             NaN          6.0          NaN        NaN     NaN
boundaries       NaN          NaN          0.0        NaN     NaN
wickets          NaN          NaN          NaN        1.0     NaN
dots             NaN          NaN          NaN        NaN     5.0

Konstas — who bowled to him:
                                  balls  runs  dots  boundaries  wickets
bowler         type_of_bowler                                           
Joel Davis     Left Off               3     2     1           0        0
Mitchell Starc Left Fast              2     2     0           0        0
Sam Curran     Left Medium Fast       4     2     2           0        0
Jack Edwards   Right Medium Fast      2     0     2           0        1

Konstas — line/length breakdown:
                             balls  runs  dots  boundaries  wickets
line        len

In [20]:
manenti = df[df['bowler'] == 'Ben Manenti']

print("Manenti — overall:")
print(manenti.agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
))

print("\nManenti — who did he bowl to:")
print(manenti.groupby(['striker', 'batter_type']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
).sort_values('runs', ascending=False))

print("\nManenti — line/length breakdown:")
print(manenti.groupby(['line', 'length']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
))

print("\nManenti — shot selection against him:")
print(manenti.groupby(['line', 'length', 'shot_played']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum')
).sort_values('balls', ascending=False).head(15))

print("\nManenti — edged or beaten:")
print(manenti[['striker', 'line', 'length', 'shot_played', 
               'runs_batter', 'edged', 'beaten', 'ball_speed']].to_string())

print("\nManenti — by phase:")
print(manenti.groupby('phase').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    dots=('is_dot', 'sum')
))

Manenti — overall:
            is_legal  runs_batter  is_boundary  is_wicket  is_dot
balls           18.0          NaN          NaN        NaN     NaN
runs             NaN         17.0          NaN        NaN     NaN
boundaries       NaN          NaN          1.0        NaN     NaN
wickets          NaN          NaN          NaN        1.0     NaN
dots             NaN          NaN          NaN        NaN     6.0

Manenti — who did he bowl to:
                            balls  runs  dots  boundaries  wickets
striker        batter_type                                        
Nic Maddinson  L                6     8     2           1        0
David Warner   L                6     6     1           0        0
Matthew Gilkes L                3     2     1           0        0
Sam Billings   R                3     1     2           0        1

Manenti — line/length breakdown:
                             balls  runs  dots  boundaries  wickets
line        length                                

In [21]:
mcandrew = df[df["bowler"] == "Nathan McAndrew"]

print("McAndrew — overall:")
print(mcandrew.agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
))

print("\nMcAndrew — who did he bowl to:")
print(mcandrew.groupby(['striker', 'batter_type']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
).sort_values('runs', ascending=False))

print("\nMcAndrew — line/length breakdown:")
print(mcandrew.groupby(['line', 'length']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    dots=('is_dot', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum')
))

print("\nMcAndrew — shot by shot:")
print(mcandrew.groupby(['line', 'length', 'shot_played']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum')
).sort_values('runs', ascending=False).head(15))

print("\nMcAndrew — full delivery detail:")
print(mcandrew[['striker', 'batter_type', 'line', 'length', 'shot_played',
                'runs_batter', 'is_boundary', 'edged', 'beaten',
                'ball_speed', 'phase']].to_string())

McAndrew — overall:
            is_legal  runs_batter  is_boundary  is_wicket  is_dot
balls           12.0          NaN          NaN        NaN     NaN
runs             NaN         33.0          NaN        NaN     NaN
boundaries       NaN          NaN          6.0        NaN     NaN
wickets          NaN          NaN          NaN        2.0     NaN
dots             NaN          NaN          NaN        NaN     3.0

McAndrew — who did he bowl to:
                           balls  runs  dots  boundaries  wickets
striker       batter_type                                        
Steve Smith   R                8    28     1           5        0
Babar Azam    R                2     4     1           1        1
Josh Phillipe R                2     1     1           0        1

McAndrew — line/length breakdown:
                             balls  runs  dots  boundaries  wickets
line        length                                                 
Leg         FT                   1     1     0     

In [22]:
# All Smith deliveries in order
smith = df[df["striker"] == "Steve Smith"].copy()
smith = smith.sort_values(['innings', 'over', 'ball']).reset_index(drop=True)

print("Smith — every delivery in sequence:")
print(smith[['over', 'ball', 'bowler', 'type_of_bowler', 'line', 'length',
             'shot_played', 'runs_batter', 'is_boundary', 'phase']].to_string())

print("\nSmith — runs by type_of_bowler:")
print(smith.groupby('type_of_bowler').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
).sort_values('runs', ascending=False))

print("\nSmith — cumulative runs over time (to see when he got going):")
smith['cumulative_runs'] = smith['runs_batter'].cumsum()
print(smith[['over', 'ball', 'bowler', 'type_of_bowler', 'runs_batter', 'cumulative_runs']].to_string())

Smith — every delivery in sequence:
    over  ball           bowler     type_of_bowler         line           length        shot_played  runs_batter  is_boundary            phase
0      0     3     Chris Green           Right Off  Outside Off  Short of length              Flick            1        False  Powerplay (1-6)
1      1     3      Ryan Hadley         Right Fast          Off      Good Length  Back Foot Defence            1        False  Powerplay (1-6)
2      1     5      Ryan Hadley         Right Fast       Middle  Short of length          Pull Shot            1        False  Powerplay (1-6)
3      2     1         Wes Agar  Right Medium Fast          Off          Bouncer          Step out             0        False  Powerplay (1-6)
4      2     2         Wes Agar  Right Medium Fast       Middle  Short of length          Pull Shot            6         True  Powerplay (1-6)
5      2     3         Wes Agar  Right Medium Fast  Outside Leg  Short of length         Square Cut       

In [23]:
# Add arm type column
df['arm_type'] = df['type_of_bowler'].apply(
    lambda x: 'Left Arm' if str(x).startswith('Left') else 'Right Arm'
)

# Thunder batting only (innings 1)
thunder_batting = df[df['innings'] == 1].copy()

print("Thunder — overall vs Left Arm vs Right Arm:")
print(thunder_batting.groupby('arm_type').agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    sixes=('is_six', 'sum'),
    fours=('is_four', 'sum'),
    wickets=('is_wicket', 'sum'),
    dots=('is_dot', 'sum')
).assign(
    sr=lambda x: round((x['runs'] / x['balls']) * 100, 1),
    economy=lambda x: round((x['runs'] / x['balls']) * 6, 2),
    dot_pct=lambda x: round((x['dots'] / x['balls']) * 100, 1)
))

print("\nThunder — each batter vs Left Arm:")
thunder_batting[thunder_batting['arm_type'] == 'Left Arm'].groupby(['striker', 'batter_type']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    dots=('is_dot', 'sum')
).assign(sr=lambda x: round((x['runs'] / x['balls']) * 100, 1)).sort_values('runs', ascending=False)

print(thunder_batting[thunder_batting['arm_type'] == 'Left Arm'].groupby(['striker', 'batter_type']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    dots=('is_dot', 'sum')
).assign(sr=lambda x: round((x['runs'] / x['balls']) * 100, 1)).sort_values('runs', ascending=False).to_string())

print("\nThunder — each batter vs Right Arm:")
print(thunder_batting[thunder_batting['arm_type'] == 'Right Arm'].groupby(['striker', 'batter_type']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    dots=('is_dot', 'sum')
).assign(sr=lambda x: round((x['runs'] / x['balls']) * 100, 1)).sort_values('runs', ascending=False).to_string())

print("\nThunder — by phase vs arm type:")
print(thunder_batting.groupby(['phase', 'arm_type']).agg(
    balls=('is_legal', 'sum'),
    runs=('runs_batter', 'sum'),
    boundaries=('is_boundary', 'sum'),
    dots=('is_dot', 'sum')
).assign(sr=lambda x: round((x['runs'] / x['balls']) * 100, 1)).to_string())

Thunder — overall vs Left Arm vs Right Arm:
           balls  runs  boundaries  sixes  fours  wickets  dots    sr  economy  dot_pct
arm_type                                                                               
Left Arm      54    63           7      2      5        4    20 116.7      7.0     37.0
Right Arm     66   115          17      5     12        2    24 174.2     10.4     36.4

Thunder — each batter vs Left Arm:
                            balls  runs  boundaries  dots    sr
striker        batter_type                                     
David Warner   L               25    27           2     9 108.0
Daniel Sams    R                4    10           2     2 250.0
Sam Billings   R                7    10           1     1 142.9
Matthew Gilkes L                4     9           2     1 225.0
Sam Konstas    R                9     6           0     3  66.7
Nic Maddinson  L                4     1           0     3  25.0
Chris Green    R                1     0           0     

In [24]:
thunder_batting = df[df["innings"] == 1].copy()
thunder_batting["arm_type"] = thunder_batting["type_of_bowler"].apply(
    lambda x: "Left Arm" if str(x).startswith("Left") else "Right Arm"
)

print("Right Arm — line/length breakdown vs Thunder:")
print(thunder_batting[thunder_batting["arm_type"] == "Right Arm"].groupby("line").agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    dots=("is_dot", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum")
).assign(sr=lambda x: round((x["runs"] / x["balls"]) * 100, 1)).sort_values("runs", ascending=False))

print("\nLeft Arm — line/length breakdown vs Thunder:")
print(thunder_batting[thunder_batting["arm_type"] == "Left Arm"].groupby("line").agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    dots=("is_dot", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum")
).assign(sr=lambda x: round((x["runs"] / x["balls"]) * 100, 1)).sort_values("runs", ascending=False))

print("\nRight Arm — shot selection vs Thunder (top 15 by runs):")
print(thunder_batting[thunder_batting["arm_type"] == "Right Arm"].groupby(["line", "shot_played"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum")
).sort_values("runs", ascending=False).head(15).to_string())

print("\nLeft Arm — shot selection vs Thunder (top 15 by runs):")
print(thunder_batting[thunder_batting["arm_type"] == "Left Arm"].groupby(["line", "shot_played"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum")
).sort_values("runs", ascending=False).head(15).to_string())

Right Arm — line/length breakdown vs Thunder:
             balls  runs  dots  boundaries  wickets    sr
line                                                     
Middle          18    37     5           5        2 205.6
Outside Leg     13    32     6           6        0 246.2
Outside Off     22    27    10           4        0 122.7
Off              6    13     1           2        0 216.7
Leg              7     6     2           0        0  85.7

Left Arm — line/length breakdown vs Thunder:
             balls  runs  dots  boundaries  wickets    sr
line                                                     
Outside Off     17    26     6           4        1 152.9
Off             10    15     2           2        0 150.0
Outside Leg     13    11     4           0        0  84.6
Middle           7     9     3           1        1 128.6
Leg              7     2     5           0        2  28.6

Right Arm — shot selection vs Thunder (top 15 by runs):
                          balls  runs  

In [25]:
# Thunder overs 8-10
thunder_8_10 = df[
    (df["innings"] == 1) &
    (df["over"].between(7, 9))
].copy()

print("Thunder — overs 8-10 delivery by delivery:")
print(thunder_8_10[["over", "ball", "striker", "bowler",
                     "type_of_bowler", "line", "length",
                     "shot_played", "runs_batter", "wicket",
                     "extras"]].to_string())

# Sixers overs 8-10
sixers_8_10 = df[
    (df["innings"] == 2) &
    (df["over"].between(7, 9))
].copy()

print("\nSixers — overs 8-10 delivery by delivery:")
print(sixers_8_10[["over", "ball", "striker", "bowler",
                    "type_of_bowler", "line", "length",
                    "shot_played", "runs_batter", "wicket",
                    "extras"]].to_string())

print("\nThunder overs 8-10 summary:")
print(thunder_8_10.agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum"),
    dots=("is_dot", "sum"),
    extras=("extras", "sum")
))

print("\nSixers overs 8-10 summary:")
print(sixers_8_10.agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum"),
    dots=("is_dot", "sum"),
    extras=("extras", "sum")
))

print("\nThunder — who bowled overs 8-10 vs Sixers:")
print(sixers_8_10.groupby(["over", "bowler", "type_of_bowler"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum")
))

Thunder — overs 8-10 delivery by delivery:
    over  ball       striker          bowler     type_of_bowler         line           length        shot_played  runs_batter wicket  extras
43     7     1   Sam Konstas      Sam Curran   Left Medium Fast          Off  Short of length  Back Foot Defence            0     No     0.0
44     7     2   Sam Konstas      Sam Curran   Left Medium Fast  Outside Leg      Good Length          Off Drive            1     No     0.0
45     7     3  David Warner      Sam Curran   Left Medium Fast       Middle  Short of length  Back Foot Defence            0     No     0.0
46     7     4  David Warner      Sam Curran   Left Medium Fast  Outside Leg  Short of length          Pull Shot            0     No     1.0
47     7     4  David Warner      Sam Curran   Left Medium Fast  Outside Off      Good Length           Late Cut            1     No     0.0
48     7     5   Sam Konstas      Sam Curran   Left Medium Fast       Middle  Short of length    Forward Defenc

In [26]:
# Over by over runs for both innings
over_summary = df.groupby(['innings', 'over']).agg(
    runs=('runs_total', 'sum'),
    wickets=('is_wicket', 'sum'),
    balls=('is_legal', 'sum'),
    boundaries=('is_boundary', 'sum')
).reset_index()

# Cumulative runs and wickets
over_summary = over_summary.sort_values(['innings', 'over'])
over_summary['cumulative_runs'] = over_summary.groupby('innings')['runs'].cumsum()
over_summary['cumulative_wickets'] = over_summary.groupby('innings')['wickets'].cumsum()

print("Over by over — both innings:")
print(over_summary[['innings', 'over', 'runs', 'wickets', 
                     'boundaries', 'cumulative_runs', 
                     'cumulative_wickets']].to_string())

# Run differential over by over (Sixers minus Thunder)
thunder_overs = over_summary[over_summary['innings'] == 1][['over', 'runs', 'cumulative_runs', 'cumulative_wickets']].rename(
    columns={'runs': 'thunder_runs', 'cumulative_runs': 'thunder_cumulative', 'cumulative_wickets': 'thunder_wickets'})

sixers_overs = over_summary[over_summary['innings'] == 2][['over', 'runs', 'cumulative_runs', 'cumulative_wickets']].rename(
    columns={'runs': 'sixers_runs', 'cumulative_runs': 'sixers_cumulative', 'cumulative_wickets': 'sixers_wickets'})

comparison = thunder_overs.merge(sixers_overs, on='over')
comparison['run_diff'] = comparison['sixers_cumulative'] - comparison['thunder_cumulative']

print("\nOver by over comparison (positive = Sixers ahead):")
print(comparison[['over', 'thunder_runs', 'sixers_runs', 
                   'thunder_cumulative', 'sixers_cumulative',
                   'thunder_wickets', 'sixers_wickets',
                   'run_diff']].to_string())

Over by over — both innings:
    innings  over  runs  wickets  boundaries  cumulative_runs  cumulative_wickets
0         1     0   3.0        0           0              3.0                   0
1         1     1  15.0        0           3             18.0                   0
2         1     2  13.0        0           3             31.0                   0
3         1     3  18.0        0           4             49.0                   0
4         1     4   5.0        0           0             54.0                   0
5         1     5   5.0        1           0             59.0                   1
6         1     6   9.0        0           1             68.0                   1
7         1     7   4.0        0           0             72.0                   1
8         1     8   4.0        1           0             76.0                   2
9         1     9   7.0        0           0             83.0                   2
10        1    10  10.0        0           1             93.0        

In [27]:
expensive = ["Wes Agar", "Nic Maddinson", "Nathan McAndrew"]

for bowler_name in expensive:
    data = df[df["bowler"] == bowler_name]
    print(f"\n{'='*60}")
    print(f"{bowler_name} — full delivery detail:")
    print(data[["innings", "over", "ball", "striker", "batter_type",
                "line", "length", "shot_played", "runs_batter",
                "is_boundary", "edged", "beaten",
                "ball_speed", "phase"]].to_string())
    
    print(f"\n{bowler_name} — who did he bowl to:")
    print(data.groupby(["striker", "batter_type"]).agg(
        balls=("is_legal", "sum"),
        runs=("runs_batter", "sum"),
        boundaries=("is_boundary", "sum"),
        wickets=("is_wicket", "sum"),
        dots=("is_dot", "sum")
    ).assign(sr=lambda x: round((x["runs"]/x["balls"])*100,1)).sort_values("runs", ascending=False).to_string())

    print(f"\n{bowler_name} — line/length breakdown:")
    print(data.groupby(["line", "length"]).agg(
        balls=("is_legal", "sum"),
        runs=("runs_batter", "sum"),
        dots=("is_dot", "sum"),
        boundaries=("is_boundary", "sum"),
        wickets=("is_wicket", "sum")
    ).assign(sr=lambda x: round((x["runs"]/x["balls"])*100,1)).sort_values("runs", ascending=False).to_string())

    print(f"\n{bowler_name} — by phase:")
    print(data.groupby("phase").agg(
        balls=("is_legal", "sum"),
        runs=("runs_batter", "sum"),
        boundaries=("is_boundary", "sum"),
        wickets=("is_wicket", "sum"),
        dots=("is_dot", "sum")
    ).assign(economy=lambda x: round((x["runs"]/x["balls"])*6,2)).to_string())


Wes Agar — full delivery detail:
     innings  over  ball      striker batter_type         line           length        shot_played  runs_batter  is_boundary edged beaten  ball_speed            phase
142        2     2     1  Steve Smith           R          Off          Bouncer          Step out             0        False    No    Yes       135.2  Powerplay (1-6)
143        2     2     2  Steve Smith           R       Middle  Short of length          Pull Shot            6         True    No     No       130.1  Powerplay (1-6)
144        2     2     3  Steve Smith           R  Outside Leg  Short of length         Square Cut            2        False    No     No       138.4  Powerplay (1-6)
145        2     2     4  Steve Smith           R          Off      Good Length    Forward Defence            0        False   Yes     No       133.5  Powerplay (1-6)
146        2     2     5  Steve Smith           R       Middle             Full              Flick            3        False    No 

In [28]:
# Thunder bowlers in the chase (innings 2)
thunder_bowling = df[df["innings"] == 2].copy()

# Total balls per bowler for pct calculation
bowler_totals = thunder_bowling.groupby("bowler")["is_legal"].sum().rename("total_balls")

print("Thunder bowlers — LINE breakdown:")
line_breakdown = thunder_bowling.groupby(["bowler", "line"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
).assign(sr=lambda x: round((x["runs"]/x["balls"])*100, 1))
line_breakdown = line_breakdown.join(bowler_totals, on="bowler")
line_breakdown["pct"] = round((line_breakdown["balls"]/line_breakdown["total_balls"])*100, 1)
print(line_breakdown.drop(columns="total_balls").sort_values(["bowler", "balls"], ascending=[True, False]).to_string())

print("\nThunder bowlers — LENGTH breakdown:")
length_breakdown = thunder_bowling.groupby(["bowler", "length"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
).assign(sr=lambda x: round((x["runs"]/x["balls"])*100, 1))
length_breakdown = length_breakdown.join(bowler_totals, on="bowler")
length_breakdown["pct"] = round((length_breakdown["balls"]/length_breakdown["total_balls"])*100, 1)
print(length_breakdown.drop(columns="total_balls").sort_values(["bowler", "balls"], ascending=[True, False]).to_string())

Thunder bowlers — LINE breakdown:
                             balls  runs  boundaries    sr  pct
bowler          line                                           
Aidan O'Connor  Outside Off      4     2           0  50.0 33.3
                Leg              3     5           1 166.7 25.0
                Middle           3     2           0  66.7 25.0
                Off              1     1           0 100.0  8.3
                Outside Leg      1     6           1 600.0  8.3
Chris Green     Outside Off      8    12           2 150.0 33.3
                Middle           7     2           0  28.6 29.2
                Off              6     4           0  66.7 25.0
                Outside Leg      2     0           0   0.0  8.3
                Leg              1     0           0   0.0  4.2
Nathan McAndrew Outside Off      6    15           3 250.0 50.0
                Off              4    11           2 275.0 33.3
                Leg              1     1           0 100.0  8.3
      

In [29]:
babar = df[df["striker"] == "Babar Azam"].copy()

print("Babar — overall:")
print(babar.agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum"),
    dots=("is_dot", "sum")
))

print("\nBabar — who bowled to him:")
print(babar.groupby(["bowler", "type_of_bowler"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    dots=("is_dot", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum")
).assign(sr=lambda x: round((x["runs"]/x["balls"])*100,1)).sort_values("runs", ascending=False).to_string())

print("\nBabar — line/length breakdown:")
print(babar.groupby(["line", "length"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    dots=("is_dot", "sum"),
    boundaries=("is_boundary", "sum"),
    wickets=("is_wicket", "sum")
).assign(sr=lambda x: round((x["runs"]/x["balls"])*100,1)).sort_values("runs", ascending=False).to_string())

print("\nBabar — shot selection (top 15 by runs):")
print(babar.groupby(["line", "length", "shot_played"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum")
).sort_values("runs", ascending=False).head(15).to_string())

print("\nBabar — by phase:")
print(babar.groupby("phase").agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
    dots=("is_dot", "sum")
).assign(sr=lambda x: round((x["runs"]/x["balls"])*100,1)).to_string())

print("\nBabar — dismissal detail:")
print(babar[babar["is_wicket"]==1][
    ["bowler", "type_of_bowler", "line", "length",
     "shot_played", "dismissal_type", "edged", "beaten", "ball_speed"]
])

Babar — overall:
            is_legal  runs_batter  is_boundary  is_wicket  is_dot
balls           39.0          NaN          NaN        NaN     NaN
runs             NaN         47.0          NaN        NaN     NaN
boundaries       NaN          NaN          7.0        NaN     NaN
wickets          NaN          NaN          NaN        1.0     NaN
dots             NaN          NaN          NaN        NaN    15.0

Babar — who bowled to him:
                                   balls  runs  dots  boundaries  wickets    sr
bowler          type_of_bowler                                                 
Tanveer Sangha  Right Leg             10    16     1           2        0 160.0
Chris Green     Right Off             15    12     9           2        0  80.0
Ryan Hadley     Right Fast             4     8     0           1        0 200.0
Aidan O'Connor  Right Medium Fast      4     5     2           1        0 125.0
Nathan McAndrew Right Medium Fast      2     4     1           1        1 200.0

In [31]:
print("Babar — shot selection (top 15 by runs):")
print(babar.groupby(["line"]).agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum")
).sort_values("runs", ascending=False).head(15).to_string())

print("\nBabar — dismissal detail:")
print(babar[babar["is_wicket"]==1][
    ["bowler", "type_of_bowler", "line", "length",
     "shot_played", "dismissal_type", "edged", "beaten", "ball_speed"]
])

print("\nBabar — by phase:")
print(babar.groupby("phase").agg(
    balls=("is_legal", "sum"),
    runs=("runs_batter", "sum"),
    boundaries=("is_boundary", "sum"),
    dots=("is_dot", "sum")
).assign(sr=lambda x: round((x["runs"]/x["balls"])*100,1)).to_string())

Babar — shot selection (top 15 by runs):
             balls  runs  boundaries
line                                
Outside Off     14    23           4
Outside Leg      6     9           2
Middle          12     8           0
Leg              5     6           1
Off              2     1           0

Babar — dismissal detail:
              bowler     type_of_bowler         line           length shot_played dismissal_type edged beaten  \
207  Nathan McAndrew  Right Medium Fast  Outside Off  Short of length   Off Drive         Bowled   Yes     No   

     ball_speed  
207       127.0  

Babar — by phase:
                 balls  runs  boundaries  dots    sr
phase                                               
Middle (7-15)       21    21           3    10 100.0
Powerplay (1-6)     18    26           4     5 144.4
